# 1. Watch a plasma wake in WarpX

A short laser pulse pushes plasma electrons aside. They oscillate behind it,
creating a wake that can accelerate electrons. A short density drop changes
the wake and can enable trapping.

This is a small **3D teaching example**, not a simulation of the real HTU
source. We will look at electron density, the accelerating field, and the
energy spectrum. It runs independently of ImpactX.

Open this notebook from the `htu` folder. Use a GPU-enabled WarpX executable
or a GPU Python kernel. Registered participants can open their AWS session
from the link sent in Slack; select **WarpX GPU** and use
`warpx_executable = None` there. The native executable below matches the local build;
change it for your machine, or set it to `None` to use the kernel's pywarpx.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
from openpmd_viewer import OpenPMDTimeSeries
from plot_wakefield import plot_density_evolution, plot_snapshot

HTU = Path.cwd()
assert (HTU / "lwfa_warpx").is_dir(), "Open this notebook from the htu folder."
local_executable = Path.home() / "Sources/WarpX/build/bin/warpx.3d"
warpx_executable = local_executable if local_executable.is_file() else None
# In the tutorial container's WarpX GPU kernel, set: warpx_executable = None
(HTU / "runs").mkdir(exist_ok=True)
run = Path(tempfile.mkdtemp(prefix="wake-", dir=HTU / "runs"))
shutil.copy2(HTU / "lwfa_warpx/lwfa_warpx_input.txt", run)
print("Outputs:", run)

## Run the simulation

The small input is `lwfa_warpx/lwfa_warpx_input.txt`. Each run gets a fresh
output directory. Progress appears below. No electron-energy cut or ImpactX
conversion is required.

In [ ]:
if warpx_executable is None:
    command = [sys.executable, str(HTU / "lwfa_warpx/run_lwfa_warpx.py")]
else:
    command = [str(warpx_executable), "lwfa_warpx_input.txt"]
env = dict(os.environ, OMP_NUM_THREADS="2")
subprocess.run(command, cwd=run, env=env, check=True)
ts = OpenPMDTimeSeries(str(run / "diags/diag1"))
print("Saved steps:", ts.iterations)

## Watch the wake evolve

These three density slices share one color scale. The horizontal coordinate
`z-ct` follows the moving window. Look for the low-density cavity and the
concentrated electrons behind it.

In [ ]:
sys.path.insert(0, str(HTU / "lwfa_warpx"))

fig = plot_density_evolution(run / "diags/diag1")
plt.show()

## Look at the wake

Change `iteration` to any saved step and rerun the cell. The middle frame
usually shows the wake inside the plasma; the final frame follows the beam
into vacuum. White contours indicate the laser field. The spectrum includes
all electrons still in the moving box, including untrapped plasma electrons.

In [ ]:
sys.path.insert(0, str(HTU / "lwfa_warpx"))

iteration = int(ts.iterations[len(ts.iterations) // 2])
fig = plot_snapshot(run / "diags/diag1", iteration)
plt.show()

## Try changing one thing

Change `a0` in the input to alter the laser strength, or set `n_up = n_down`
to remove the injection downramp. Rerun from the setup cell for a fresh result.
Compare the density cavity, longitudinal field and spectrum at similar times.
A high-energy tail alone is not proof of a useful trapped bunch.

Continue with the separate [ImpactX example](htu_transport.ipynb).
Optional [coupling instructions](README.md#optional-coupling) explain what a
valid WarpX-to-ImpactX transfer would require.